# Delta-Ripple Sequence Coupling: First Hour of Postsleep

First hour of postsleep only, same analysis as `Task2_delta_ripple_sequence.ipynb`.

Rates use NREM minutes within hour 1 as denominator.

---
# Setup

## Configuration

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# lab repo root on the path, so `modules` imports work from this folder
REPO_ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "modules" / "project_config.py").exists()
)
sys.path.insert(0, str(REPO_ROOT))
from modules.project_config import get_path  # noqa: E402

# local paths come from config.yaml (see config.example.yaml)
ROOT = get_path("R1_8_root")  # holds Rat_HM_Ephys_TD_Analysis_R1_8 and _R9_16
assert get_path("R9_16_root") == ROOT, "these notebooks expect R1_8_root == R9_16_root"
WAVE_CACHE = get_path("event_cache_root")
SESSION_MAP_CSV = get_path("session_map_csv")
DAY_GRID_XLSX = get_path("day_grid_xlsx")
OUT = REPO_ROOT / "results" / "leticia_event_coupling"  # figures and caches
SPINDLE_METHOD = "wavelet_amp_1_ampcore_3"
PHASES = ["postsleep"]

BOUT_KEYS = ["rat", "date", "trial", "nrem_bout_index"]
AREAS = ["PL", "RSC", "HPC"]
RATS = [1, 3, 7, 8, 11, 12, 13]

# 1000 iterations, same as the full dataset
N_ITER = 1000
SHIFT_RATS = RATS  # all seven rats (full data),

MEASURES = ["rate_RD", "ratio_RD", "prop_ripples", "rate_DR", "ratio_DR"]
MIN_PAIRS = 0  # no threshold: see the NREM-duration cutoff below

FIGDIR = OUT / "task2_delta_ripple_first_hour_figures"
FIGDIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name):
    fig.savefig(FIGDIR / f"{name}.png", dpi=200, bbox_inches="tight")
    print("saved:", name)


print("drive reachable:", ROOT.exists(), " (False is fine - we load from WAVE_CACHE)")
print("rats:", RATS, "| circular shift on:", SHIFT_RATS, "|", N_ITER, "iterations")

## Load

Bad days are removed in `build_wave` (`excluded_days.py`).

In [ ]:
from modules.event_coupling.build_tables import add_condition, build_wave

ripples = add_condition(
    build_wave(ROOT, "ripple", SPINDLE_METHOD, PHASES, WAVE_CACHE, rebuild=False),
    SESSION_MAP_CSV,
)
deltas = add_condition(
    build_wave(ROOT, "delta", SPINDLE_METHOD, PHASES, WAVE_CACHE, rebuild=False),
    SESSION_MAP_CSV,
)

rip = ripples[(ripples["rat"].isin(RATS)) & (ripples["group"] != "other")]
dlt = deltas[
    (deltas["rat"].isin(RATS))
    & (deltas["group"] != "other")
    & (deltas["area"].isin(AREAS))
]

print(
    f"{len(rip):,} ripples | {len(dlt):,} deltas  (full postsleep, before first-hour restriction)"
)
print(rip.groupby("group").size().to_string())

### Check that the bad days were dropped

In [ ]:
from modules.event_coupling.excluded_days import excluded_table

ex = excluded_table()
check = ex.merge(
    rip[["rat", "date"]]
    .assign(date=pd.to_datetime(rip["date"]).dt.normalize())
    .drop_duplicates(),
    on=["rat", "date"],
)
print(
    f"{len(ex)} rat-days on the exclusion list | still present in the data: {len(check)} (want 0)"
)
assert len(check) == 0, "exclusion did not fire - is excluded_days.py in this folder?"

## Time labels

In [ ]:
from modules.event_coupling.gl_sessions import add_gl_sessions, build_gl_sessions
from modules.event_coupling.hc_sessions import add_session_type, build_day_table


def prep_keys(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["rat"] = df["rat"].astype("int64")
    return df


gl = prep_keys(build_gl_sessions(ROOT, cache_dir=WAVE_CACHE, rebuild=False))
days = add_session_type(build_day_table(DAY_GRID_XLSX, verbose=False))


def add_context(df):
    df = add_gl_sessions(prep_keys(df), gl)
    return df.merge(
        prep_keys(days)[["rat", "date", "session_type"]].drop_duplicates(),
        on=["rat", "date"],
        how="left",
    )


rip_ctx = add_context(rip)
dlt_ctx = add_context(dlt)

# days since each rat's first recording
first = rip_ctx.groupby("rat")["date"].transform("min")
rip_ctx["days_elapsed"] = (rip_ctx["date"] - first).dt.days
dlt_ctx = dlt_ctx.merge(
    rip_ctx[["rat", "date", "days_elapsed"]].drop_duplicates(),
    on=["rat", "date"],
    how="left",
)

print(
    rip_ctx.groupby(["rat", "group"])["date"]
    .nunique()
    .unstack(fill_value=0)
    .to_string()
)

## Restrict to the first hour of postsleep

`peak_index` restarts in every ~30-min trial file, so elapsed time = trial position x 30 min + within-trial index (`TRIAL_LEN = 1,800,000`).

`day_nrem_hour1` = NREM minutes per rat-day within hour 1, used as the rate denominator.

In [ ]:
TRIAL_LEN = 1_800_000  # 30 min at 1000Hz, confirmed from real trial lengths
# (same recording system as cross_regional_first_hour.ipynb)


def add_elapsed_time(df, index_col):
    """True elapsed time since the start of postsleep, correcting for
    peak_index resetting to 0 at the start of EACH trial file. Uses
    whichever trial number is lowest for that rat-day as the start of
    postsleep -- handles both normal split-trial days and the unsplit
    early-2022 days (trial=1) without a special case. Validated on
    synthetic data in cross_regional_first_hour.ipynb."""
    min_trial = df.groupby(["rat", "date"])["trial"].transform("min")
    return (df["trial"] - min_trial) * TRIAL_LEN + df[index_col]


def split_bout_by_hour(start, end):
    """Minutes of NREM in each of hours 1-4 for one bout [start, end),
    given elapsed-time-corrected start/end sample indices."""
    HOUR = 60 * 60 * 1000
    out = np.zeros(4)
    for h in range(4):
        lo, hi = h * HOUR, (h + 1) * HOUR
        overlap = max(0, min(end, hi) - max(start, lo))
        out[h] = overlap / 1000 / 60
    return out


ONE_HOUR_SAMPLES = 60 * 60 * 1000  # 1000 Hz -> 1 sample index = 1 ms


def filter_first_hour_fixed(df, index_col):
    """Correct version: filters on true elapsed time (trial position x 30
    min + within-trial index), not raw peak_index."""
    df = df.copy()
    df["_elapsed"] = add_elapsed_time(df, index_col)
    return df[df["_elapsed"] < ONE_HOUR_SAMPLES].drop(columns="_elapsed")


# check that the first-hour correction looks right
bouts_check = rip_ctx[
    ["rat", "date", "trial", "nrem_bout_start_index", "nrem_bout_end_index"]
].drop_duplicates()
bouts_check["elapsed_start"] = add_elapsed_time(bouts_check, "nrem_bout_start_index")
bouts_check["elapsed_end"] = add_elapsed_time(bouts_check, "nrem_bout_end_index")
hour_mins = np.array(
    [
        split_bout_by_hour(s, e)
        for s, e in zip(bouts_check["elapsed_start"], bouts_check["elapsed_end"])
    ]
)
bouts_check[["hour1", "hour2", "hour3", "hour4"]] = hour_mins
per_rat_hours = bouts_check.groupby("rat")[["hour1", "hour2", "hour3", "hour4"]].sum()

fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(len(per_rat_hours))
colors = ["#2c5f8a", "#5b9bd5", "#a9c9e8", "#d6e4f0"]
for i, hour_col in enumerate(["hour1", "hour2", "hour3", "hour4"]):
    ax.bar(
        per_rat_hours.index.astype(str),
        per_rat_hours[hour_col],
        bottom=bottom,
        label=f"Hour {i + 1}" if i < 3 else "Last hour",
        color=colors[i],
    )
    bottom += per_rat_hours[hour_col].values
ax.set_xlabel("Rat")
ax.set_ylabel("NREM duration (minutes)")
ax.set_title("Postsleep NREM duration by rat and recording hour (7 rats, ripple bouts)")
ax.legend(title="Recording hour")
plt.tight_layout()
save_fig(fig, "00_nrem_by_hour_check")
plt.show()

# --- first-hour restriction, applied to both event tables ------
rip_full, dlt_full = rip_ctx, dlt_ctx
rip_ctx = filter_first_hour_fixed(rip_full, "ripple_peak_index")
dlt_ctx = filter_first_hour_fixed(dlt_full, "delta_peak_index")

for name, full, hr in [("ripples", rip_full, rip_ctx), ("deltas", dlt_full, dlt_ctx)]:
    print(
        f"{name}: {len(full):,} total -> {len(hr):,} in true first hour "
        f"({100 * len(hr) / len(full):.1f}%)"
    )

# --- NREM minutes clipped to hour 1 (rate denominator) ------------
bouts_q = rip_full[
    ["rat", "date", "trial", "nrem_bout_start_index", "nrem_bout_end_index"]
].drop_duplicates()
bouts_q["elapsed_start"] = add_elapsed_time(bouts_q, "nrem_bout_start_index")
bouts_q["elapsed_end"] = add_elapsed_time(bouts_q, "nrem_bout_end_index")
bouts_q["hour1_min"] = [
    split_bout_by_hour(s, e)[0]
    for s, e in zip(bouts_q["elapsed_start"], bouts_q["elapsed_end"])
]
day_nrem_hour1 = bouts_q.groupby(["rat", "date"])["hour1_min"].sum()

print(
    "hour-1 NREM minutes per rat-day: median "
    f"{day_nrem_hour1.median():.1f}, min {day_nrem_hour1.min():.1f}, "
    f"max {day_nrem_hour1.max():.1f}"
)

# Per-day measures

## Five measures for every rat-day (first hour only)

Same measures as the full dataset, rates divided by `day_nrem_hour1`.

In [ ]:
from modules.event_coupling import lag_histogram as lh
from modules.event_coupling.coupling_unique import count_sequences_unique

edges = lh.DEFAULT_EDGES
left = edges[:-1]


def per_day(dlt_df, rip_df, area, min_pairs=MIN_PAIRS):
    """Five measures for each rat x recording day, one delta area.
    nrem_min comes from day_nrem_hour1 (hour-1-clipped), not the day's full
    postsleep NREM duration -- see Step 3b."""
    d = dlt_df[dlt_df["area"] == area]
    rows = []
    for (rat, date), dg in d.groupby(["rat", "date"]):
        rg = rip_df[(rip_df["rat"] == rat) & (rip_df["date"] == date)]
        if rg.empty:
            continue

        lg = lh.collect_lags(dg, rg, "delta", "ripple")
        if len(lg) < max(min_pairs, 1):
            continue
        res = lh.summarise(lg, edges, verbose=False)
        w_rd = res["windows"]["R->D"]  # ripple leads, 50-250 ms
        w_dr = res["windows"]["D->R"]  # delta leads,  50-400 ms

        nrem_min = day_nrem_hour1.get((rat, date), np.nan)
        if not nrem_min or nrem_min <= 0 or np.isnan(nrem_min):
            continue

        u = count_sequences_unique(rg, dg, "ripple", "delta")
        n_coupled = u["n_sequences"].sum()
        n_rip_tot = u["n_ripple"].sum()

        rows.append(
            {
                "rat": rat,
                "date": date,
                "area": area,
                "rate_RD": w_rd["observed"] / nrem_min,
                "ratio_RD": w_rd["ratio"],
                "n_seq": w_rd["observed"],
                "prop_ripples": n_coupled / n_rip_tot if n_rip_tot else np.nan,
                "n_ripple_coupled": int(n_coupled),
                "rate_DR": w_dr["observed"] / nrem_min,
                "ratio_DR": w_dr["ratio"],
                "n_seq_DR": w_dr["observed"],
                "pairs": len(lg),
                "n_ripple": len(rg),
                "nrem_min": nrem_min,
                "peak_ms": res["peak_ms"],
                "group": dg["group"].iloc[0],
                "session_type": dg["session_type"].iloc[0],
                "gl_num": dg["gl_num"].iloc[0],
                "session_num": dg["session_num"].iloc[0],
                "days_elapsed": dg["days_elapsed"].iloc[0],
            }
        )
    return pd.DataFrame(rows)


daily = pd.concat([per_day(dlt_ctx, rip_ctx, a) for a in AREAS], ignore_index=True)

print(f"{len(daily)} rat-day-area values (first hour only)")
print("sequences per recording day (first hour):")
print(
    daily.groupby(["area", "group"])
    .agg(
        days=("date", "nunique"),
        rats=("rat", "nunique"),
        seq_RD_median=("n_seq", "median"),
        seq_RD_min=("n_seq", "min"),
        coupled_ripples_median=("n_ripple_coupled", "median"),
        seq_DR_median=("n_seq_DR", "median"),
        ripples_median=("n_ripple", "median"),
    )
    .round(0)
    .to_string()
)

print("measures by area:")
stats = daily.groupby("area")[MEASURES].agg(["median", "mean", "std"]).round(3)
print(stats.to_string())

print("measures by area and condition:")
for m in MEASURES:
    print(f"\n--- {m} ---")
    print(
        daily.groupby(["area", "group"])[m]
        .agg(median="median", mean="mean", std="std", n="size")
        .round(3)
        .to_string()
    )

In [ ]:
print("total sequences across the first-hour dataset:")
tot = daily.groupby("area").agg(
    rat_days=("date", "size"),
    seq_RD=("n_seq", "sum"),
    coupled_ripples=("n_ripple_coupled", "sum"),
    seq_DR=("n_seq_DR", "sum"),
    ripples=("n_ripple", "sum"),
    nrem_hours=("nrem_min", lambda s: s.sum() / 60),
)
print(tot.round(0).to_string())

print("and split by condition:")
print(
    daily.groupby(["area", "group"])
    .agg(
        rat_days=("date", "size"),
        seq_RD=("n_seq", "sum"),
        coupled_ripples=("n_ripple_coupled", "sum"),
        ripples=("n_ripple", "sum"),
    )
    .round(0)
    .to_string()
)

# Pooled coupling result (first hour)

In [ ]:
fig = plt.figure(figsize=(10, 4.5))
pooled = {}
for area, colour, style in [
    ("PL", "tab:orange", "-"),
    ("RSC", "tab:purple", "-"),
    ("HPC", "tab:green", "--"),
]:
    lg = lh.collect_lags(dlt_ctx[dlt_ctx["area"] == area], rip_ctx, "delta", "ripple")
    r = lh.summarise(lg, edges, label=area)
    pooled[area] = r
    plt.plot(
        left,
        r["counts"],
        style,
        color=colour,
        lw=1.8,
        label=f"{area}  ({r['windows']['R->D']['ratio']:.2f}x chance)",
    )

plt.axvspan(-250, -50, color="tab:blue", alpha=0.10, label="R->D window")
plt.axvline(0, color="k", lw=1)
plt.xlabel("ripple peak - delta peak (ms)   [negative = ripple first]")
plt.ylabel("count")
plt.title("Delta-ripple lag histogram, all 7 rats pooled -- first hour only")
plt.legend()
plt.tight_layout()
save_fig(fig, "10_histogram_by_area")
plt.show()

In [ ]:
for area in AREAS:
    rats = sorted(rip_ctx["rat"].unique())
    ncol = 4
    nrow = int(np.ceil(len(rats) / ncol))
    fig, axes = plt.subplots(
        nrow, ncol, figsize=(4.2 * ncol, 3.2 * nrow), sharey=False, squeeze=False
    )
    for ax, rat in zip(axes.ravel(), rats):
        lg = lh.collect_lags(
            dlt_ctx[(dlt_ctx["area"] == area) & (dlt_ctx["rat"] == rat)],
            rip_ctx[rip_ctx["rat"] == rat],
            "delta",
            "ripple",
        )
        if len(lg) < 100:  # lower minimum than the 500 in the full dataset, the first
            # hour has far fewer pairs per rat to begin with
            ax.set_title(f"rat {rat}: too few pairs")
            continue
        r = lh.summarise(lg, edges, verbose=False)
        ax.plot(left, r["counts"], lw=1.4, color="tab:purple")
        ax.axvline(0, color="k", lw=0.8)
        ax.set_title(
            f"rat {rat}   peak {r['peak_ms']} ms   "
            f"{r['windows']['R->D']['ratio']:.2f}x chance",
            fontsize=9,
        )
        ax.set_xlabel("lag (ms)")
    for ax in axes.ravel()[len(rats) :]:
        ax.axis("off")
    for row in axes:
        row[0].set_ylabel("count")
    fig.suptitle(f"{area} — lag histogram per rat (first hour)", fontsize=13)
    plt.tight_layout()
    save_fig(fig, f"11_histogram_perrat_{area}")
    plt.show()

In [ ]:
from modules.event_coupling.coupling_unique import count_sequences_unique

for area in AREAS:
    d = dlt_ctx[dlt_ctx["area"] == area]
    u = count_sequences_unique(d, rip_ctx, "delta", "ripple")
    coupled = u["n_sequences"].sum()
    total = u["n_delta"].sum()
    print(
        f"{area:>4}: {coupled:>9,} of {total:>10,} delta waves "
        f"({100 * coupled / total:5.1f}%)"
    )

# Significance test (1000 iterations, with checkpoints, first hour)

In [ ]:
def plot_with_significance(dist, edges, area, window_ms, label=None, ax=None):
    """Lag histogram in absolute counts, with a significance bar over the
    tested window when the observed sum beats the shuffled null."""
    left = edges[:-1]
    real = dist["real"]
    null = dist["null"]
    mu, sd = null.mean(axis=0), null.std(axis=0)

    res = window_significance(dist, edges, window_ms)

    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(9, 4.5))

    ax.bar(left, real, width=20, align="edge", color="0.35", zorder=2)
    ax.plot(left, mu, color="0.5", lw=1, zorder=3, label="null mean")
    ax.fill_between(
        left,
        mu - 2 * sd,
        mu + 2 * sd,
        color="tab:orange",
        alpha=0.25,
        zorder=1,
        label="null \u00b12 SD",
    )
    ax.axvline(0, color="k", lw=1)

    lo, hi = window_ms
    if res["significant"]:
        y_bar = real.max() * 1.08
        ax.plot([lo, hi], [y_bar, y_bar], color="tab:red", lw=3, zorder=4)
        p_str = "p < 0.001" if res["p"] < 0.001 else f"p = {res['p']:.3f}"
        ax.text(
            (lo + hi) / 2,
            y_bar * 1.03,
            p_str,
            ha="center",
            fontsize=11,
            color="tab:red",
            fontweight="bold",
        )
        ax.set_ylim(top=y_bar * 1.25)
    else:
        ax.axvspan(lo, hi, color="tab:blue", alpha=0.08, zorder=0)
        ax.text(
            (lo + hi) / 2,
            real.max() * 1.05,
            "n.s.",
            ha="center",
            fontsize=10,
            color="tab:blue",
        )

    ax.set_xlabel("ripple peak - delta peak (ms)   [negative = ripple first]")
    ax.set_ylabel("count")
    ax.set_title(label or area)
    ax.legend(fontsize=8, loc="upper right")

    if own_fig:
        plt.tight_layout()
        return fig, res
    return res

In [ ]:
from modules.event_coupling import surrogate as sg
from modules.event_coupling.surrogate import window_significance

In [ ]:
import pickle
from pathlib import Path

CACHE_SIG = OUT / "significance_cache_first_hour"
CACHE_SIG.mkdir(parents=True, exist_ok=True)
R_TO_D_WINDOW = (-250, -50)
D_TO_R_WINDOW = (50, 400)
N_ITER_SIG = 1000
CHECKPOINT_EVERY = 200


def run_with_checkpoints(area):
    final_f = CACHE_SIG / f"dist_{area}_n{N_ITER_SIG}.pkl"
    ckpt_f = CACHE_SIG / f"dist_{area}_n{N_ITER_SIG}_partial.pkl"

    if final_f.exists():
        print(f"{area}: loaded finished result from cache")
        return pickle.load(open(final_f, "rb"))

    resume = None
    if ckpt_f.exists():
        resume = pickle.load(open(ckpt_f, "rb"))
        print(f"{area}: found checkpoint at {resume['n_done']}/{N_ITER_SIG}, resuming")

    def save_checkpoint(partial):
        pickle.dump(partial, open(ckpt_f, "wb"))
        print(f"    [checkpoint saved: {partial['n_done']}/{N_ITER_SIG}]")

    dist = sg.surrogate_distribution(
        dlt_ctx[dlt_ctx["area"] == area],
        rip_ctx,
        "delta",
        "ripple",
        edges,
        n_iter=N_ITER_SIG,
        seed=0,
        scope="recording",
        verbose=True,
        checkpoint_every=CHECKPOINT_EVERY,
        checkpoint_fn=save_checkpoint,
        resume_from=resume,
    )

    pickle.dump(dist, open(final_f, "wb"))
    if ckpt_f.exists():
        ckpt_f.unlink()
    print(f"{area}: computed and cached")
    return dist


sig_results = {}
fig, axes = plt.subplots(1, 3, figsize=(19, 4.5))
for ax, area in zip(axes, ["PL", "RSC", "HPC"]):
    dist = run_with_checkpoints(area)
    res = plot_with_significance(dist, edges, area, R_TO_D_WINDOW, label=area, ax=ax)
    sig_results[area] = res

plt.tight_layout()
save_fig(fig, "12_significance_by_area")
plt.show()

for area, r in sig_results.items():
    print(
        f"{area}: observed {r['observed']:,} | null mean {r['null_mean']:,.0f} "
        f"+/- {r['null_sd']:,.0f} | p = {r['p']:.4f} | significant = {r['significant']}"
    )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 4.5))
for ax, area in zip(axes, ["PL", "RSC", "HPC"]):
    dist = run_with_checkpoints(area)
    res = plot_with_significance(dist, edges, area, D_TO_R_WINDOW, label=area, ax=ax)

plt.tight_layout()
save_fig(fig, "14_significance_DtoR_by_area")
plt.show()

In [ ]:
def plot_ratio_comparison(dist, edges, area, ax=None):
    """R->D vs D->R, shown as ratio-to-chance rather than a significance
    verdict -- same rationale as the full-dataset notebook (null SD is tiny
    relative to its mean at this sample size, so the binary flag stops being
    informative)."""
    left = edges[:-1]
    real = dist["real"]
    null = dist["null"]
    mu, sd = null.mean(axis=0), null.std(axis=0)

    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(9, 4.5))

    ax.bar(left, real, width=20, align="edge", color="0.35", zorder=2)
    ax.plot(left, mu, color="0.5", lw=1, zorder=3, label="null mean")
    ax.fill_between(
        left,
        mu - 2 * sd,
        mu + 2 * sd,
        color="tab:orange",
        alpha=0.25,
        zorder=1,
        label="null \u00b12 SD",
    )
    ax.axvline(0, color="k", lw=1)

    for lo, hi, label, colour in [
        (-250, -50, "R\u2192D", "tab:red"),
        (50, 400, "D\u2192R", "tab:blue"),
    ]:
        r = window_significance(dist, edges, (lo, hi))
        ratio = r["observed"] / r["null_mean"]
        y_bar = real.max() * (1.15 if colour == "tab:red" else 1.05)
        ax.plot([lo, hi], [y_bar, y_bar], color=colour, lw=3, zorder=4)
        ax.text(
            (lo + hi) / 2,
            y_bar * 1.02,
            f"{label}: {ratio:.2f}x chance",
            ha="center",
            fontsize=10,
            color=colour,
            fontweight="bold",
        )

    ax.set_ylim(top=real.max() * 1.3)
    ax.set_xlabel("ripple peak - delta peak (ms)   [negative = ripple first]")
    ax.set_ylabel("count")
    ax.set_title(area)
    ax.legend(fontsize=8, loc="upper left")

    if own_fig:
        plt.tight_layout()
        return fig
    return None


fig, axes = plt.subplots(1, 3, figsize=(19, 4.5))
for ax, area in zip(axes, ["PL", "RSC", "HPC"]):
    dist = run_with_checkpoints(area)
    plot_ratio_comparison(dist, edges, area, ax=ax)

plt.tight_layout()
save_fig(fig, "14_ratio_comparison_by_area")
plt.show()

for area in ["PL", "RSC", "HPC"]:
    dist = run_with_checkpoints(area)
    rd = window_significance(dist, edges, (-250, -50))
    dr = window_significance(dist, edges, (50, 400))
    print(
        f"{area}: R\u2192D {rd['observed'] / rd['null_mean']:.2f}x  |  "
        f"D\u2192R {dr['observed'] / dr['null_mean']:.2f}x"
    )

In [ ]:
rows = []
for area in AREAS:
    dist = run_with_checkpoints(area)
    for lo, hi, label in [(-250, -50, "R->D"), (50, 400, "D->R")]:
        r = window_significance(dist, edges, (lo, hi))
        rows.append(
            {
                "area": area,
                "direction": label,
                "observed": r["observed"],
                "null_mean": round(r["null_mean"]),
                "ratio": round(r["observed"] / r["null_mean"], 3),
                "p": r["p"],
                "significant": r["significant"],
            }
        )

sig_summary = pd.DataFrame(rows)
print(sig_summary.to_string(index=False))

# End-of-delta -> start-of-ripple (first hour)

D->R only, as in the full dataset.

`lag = ripple_start - delta_end`: lag < 0 overlap, lag >= 0 sequential.

In [ ]:
MAX_LAG_MS_ES = 1600
EDGES_ES = np.arange(-1600, 1620, 20)
BASELINE_ES = 1400
left_es = EDGES_ES[:-1]

res_end_start_dr = {}
for area in AREAS:
    lg = lh.collect_lags_cols(
        dlt_ctx[dlt_ctx["area"] == area],
        rip_ctx,
        "delta_end_index",
        "ripple_start_index",
        max_lag_ms=MAX_LAG_MS_ES,
    )
    res_end_start_dr[area] = lh.summarise(
        lg,
        EDGES_ES,
        baseline_ms=BASELINE_ES,
        windows=[
            ("overlap (lag<0)", -MAX_LAG_MS_ES, 0),
            ("sequential (lag>=0)", 0, MAX_LAG_MS_ES),
        ],
        label=f"{area} delta end - ripple start (first hour)",
    )

for area, r in res_end_start_dr.items():
    ov = r["windows"]["overlap (lag<0)"]["observed"]
    sq = r["windows"]["sequential (lag>=0)"]["observed"]
    print(
        f"{area}: {ov:,} pairs overlap (lag<0) | {sq:,} pairs sequential (lag>=0) "
        f"| {100 * sq / (ov + sq):.1f}% sequential"
    )

In [ ]:
for area, r in res_end_start_dr.items():
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(left_es, r["counts"], color="tab:purple", lw=1.6)
    ax.axhline(r["baseline"], color="k", lw=1, ls=":", label="baseline")
    ax.axvline(0, color="k", lw=1.2)
    ax.axvspan(
        EDGES_ES[0], 0, color="tab:red", alpha=0.06, label="overlap (already counted)"
    )
    ax.axvspan(0, EDGES_ES[-1], color="tab:green", alpha=0.06, label="sequential (new)")
    ax.set_xlabel(
        "ripple start - delta end (ms)   [<0 = overlapping, >=0 = sequential]"
    )
    ax.set_ylabel("count")
    ax.set_title(f"{area} delta end -> ripple start -- first hour")
    ax.legend(fontsize=8)
    plt.tight_layout()
    save_fig(fig, f"15_histogram_end_start_{area}")
    plt.show()

# Delta phase (Hilbert transform), not run yet

Ripple times are already restricted to hour 1.

Delta band 0.5-4 Hz (confirmed with Yixiao). The LFP is read one trial file at a time from `PreprocessedData` (peak indices are within-trial), on the channel the delta waves were detected on.

In [ ]:
import io
import shutil
import time as _time

import scipy.io
from scipy.signal import butter, hilbert, sosfiltfilt

FS_HZ = 1000.0  # 1 sample = 1 ms
DELTA_BAND = (0.5, 4.0)  # Hz (confirmed with Yixiao)

# ROOT / Rat_HM_Ephys_TD_Analysis_R1_8-or-R9_16 / <rat-group> / PreprocessedData /
#   <area> / <rat> / <date> / <phase> / chan<channel>_<trial>.mat   (unsplit days: chan<channel>.mat)
ANALYSIS_GROUPS = {
    "Rat_HM_Ephys_TD_Analysis_R1_8": ["R1-4", "R5-8"],
    "Rat_HM_Ephys_TD_Analysis_R9_16": ["R9-12", "R13-16"],
}


def _find_preproc_rat_dir(area, rat):
    """PreprocessedData/<area>/<rat>, trying every ANALYSIS_GROUPS folder."""
    for analysis, subgroups in ANALYSIS_GROUPS.items():
        for sub in subgroups:
            cand = ROOT / analysis / sub / "PreprocessedData" / area / str(rat)
            if cand.exists():
                return cand
    return None


def channel_for(delta_df, rat, area, date):
    """Recording channel of this rat/area/date's delta detection (from the event table)."""
    rows = delta_df[
        (delta_df["rat"] == rat)
        & (delta_df["area"] == area)
        & (delta_df["date"] == date)
    ]
    chans = rows["channel"].dropna().unique()
    if len(chans) == 0:
        return None
    if len(chans) > 1:
        print(
            f"    warning: rat {rat} {area} {pd.Timestamp(date).date()} has "
            f"several channels {list(chans)}, using {chans[0]}"
        )
    return int(chans[0])


class UnreadableMatFile(FileNotFoundError):
    """The .mat file exists but could not be read (network / drive problem)."""


unreadable_lfp = []  # files skipped after all retries


def _load_mat_signal(
    path, chunk_bytes=8 * 1024 * 1024, attempts=6, waits=(10, 20, 30, 40, 60)
):
    """'data' array of one .mat file. Reading straight off Z: with loadmat(path)
    failed with OSError [Errno 22] part-way through a file, so the file is copied
    into memory in 8 MB pieces first, and a failed read is retried."""
    last = None
    for attempt in range(1, attempts + 1):
        try:
            buf = io.BytesIO()
            with open(path, "rb") as f:
                shutil.copyfileobj(f, buf, chunk_bytes)
            buf.seek(0)
            return scipy.io.loadmat(buf)["data"].squeeze()
        except OSError as e:
            last = e
            if attempt < attempts:
                _time.sleep(waits[min(attempt - 1, len(waits) - 1)])
    raise UnreadableMatFile(f"{path} unreadable after {attempts} attempts: {last}")


def load_trial_lfp(rat, date, trial, area, channel, phase="postsleep"):
    """LFP of one trial ('data' in chan<channel>_<trial>.mat, or chan<channel>.mat
    for unsplit days). None if the file isn't there."""
    rat_dir = _find_preproc_rat_dir(area, rat)
    if rat_dir is None:
        return None
    phase_dir = rat_dir / pd.to_datetime(date).strftime("%Y%m%d") / phase
    for f in (
        phase_dir / f"chan{channel}_{int(trial)}.mat",
        phase_dir / f"chan{channel}.mat",
    ):
        if f.exists():
            try:
                return _load_mat_signal(f).ravel()
            except UnreadableMatFile as e:
                unreadable_lfp.append(str(e))
                return None
    return None


def bandpass_delta(signal, fs=FS_HZ, band=DELTA_BAND, order=4):
    """Zero-phase Butterworth band-pass into the delta band. Second-order sections:
    the (b, a) form of this filter is unstable at 1000 Hz (output blows up)."""
    sos = butter(order, band, btype="band", fs=fs, output="sos")
    return sosfiltfilt(sos, signal)


def delta_phase_at_times(signal, fs, event_sample_indices):
    """Delta phase (radians, -pi..pi) at each sample index; NaN outside the trace."""
    phase = np.angle(hilbert(bandpass_delta(signal, fs=fs)))
    out = np.full(len(event_sample_indices), np.nan)
    ok = (event_sample_indices >= 0) & (event_sample_indices < len(phase))
    out[ok] = phase[event_sample_indices[ok]]
    return out


def event_phases(ev_df, peak_col, area, delta_df):
    """Delta phase in `area` at every event peak, one trial file at a time
    (peak indices are within-trial sample indices)."""
    out = []
    for (rat, date, trial), g in ev_df.groupby(["rat", "date", "trial"]):
        channel = channel_for(delta_df, rat, area, date)
        if channel is None:
            continue
        sig = load_trial_lfp(rat, date, trial, area, channel)
        if sig is None:
            continue
        peaks = g[peak_col].dropna().to_numpy().astype(int)
        if len(peaks):
            out.append(delta_phase_at_times(sig, FS_HZ, peaks))
    ph = np.concatenate(out) if out else np.array([])
    return ph[~np.isnan(ph)]


def event_phase_diffs(ev_df, peak_col, area_a, area_b, delta_df):
    """Delta phase difference area_a - area_b (wrapped to -pi..pi) at every event peak."""
    out = []
    for (rat, date, trial), g in ev_df.groupby(["rat", "date", "trial"]):
        ch_a = channel_for(delta_df, rat, area_a, date)
        ch_b = channel_for(delta_df, rat, area_b, date)
        if ch_a is None or ch_b is None:
            continue
        sig_a = load_trial_lfp(rat, date, trial, area_a, ch_a)
        sig_b = load_trial_lfp(rat, date, trial, area_b, ch_b)
        if sig_a is None or sig_b is None:
            continue
        peaks = g[peak_col].dropna().to_numpy().astype(int)
        if len(peaks):
            d = delta_phase_at_times(sig_a, FS_HZ, peaks) - delta_phase_at_times(
                sig_b, FS_HZ, peaks
            )
            out.append((d[~np.isnan(d)] + np.pi) % (2 * np.pi) - np.pi)
    return np.concatenate(out) if out else np.array([])

In [ ]:
def rayleigh_test(phases):
    """Rayleigh test for non-uniformity of a circular distribution.
    Standard large-sample approximation (Zar, Biostatistical Analysis)."""
    n = len(phases)
    if n == 0:
        return np.nan, np.nan, np.nan
    C = np.sum(np.cos(phases))
    S = np.sum(np.sin(phases))
    R = np.sqrt(C**2 + S**2) / n
    z = n * R**2
    p = np.exp(-z) * (
        1
        + (2 * z - z**2) / (4 * n)
        - (24 * z - 132 * z**2 + 76 * z**3 - 9 * z**4) / (288 * n**2)
    )
    return R, z, max(min(p, 1.0), 0.0)


def circular_hist(ax, phases, bins=24, color="tab:purple"):
    counts, edges = np.histogram(phases, bins=bins, range=(-np.pi, np.pi))
    width = edges[1] - edges[0]
    ax.bar(
        edges[:-1],
        counts,
        width=width,
        align="edge",
        color=color,
        alpha=0.75,
        edgecolor="white",
    )
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)


# delta phase at each ripple peak, per delta area
phase_by_area = {}
for area in AREAS:
    phase_by_area[area] = event_phases(rip_ctx, "ripple_peak_index", area, dlt_ctx)
    print(f"{area}: {len(phase_by_area[area])} ripple-phase samples collected")

In [ ]:
fig, axes = plt.subplots(
    1, len(AREAS), figsize=(4.5 * len(AREAS), 4.5), subplot_kw={"projection": "polar"}
)
axes = np.atleast_1d(axes)
for ax, area in zip(axes, AREAS):
    phases = phase_by_area[area]
    circular_hist(ax, phases)
    if len(phases):
        R, z, p = rayleigh_test(phases)
        ax.set_title(f"{area}\nRayleigh R={R:.3f}, p={p:.4f}", fontsize=10)
    else:
        ax.set_title(f"{area}\n(no data yet)", fontsize=10)
plt.tight_layout()
save_fig(fig, "16_delta_phase_per_area")
plt.show()

In [ ]:
from itertools import combinations

cross_area_diffs = {}
for area_a, area_b in combinations(AREAS, 2):
    cross_area_diffs[(area_a, area_b)] = event_phase_diffs(
        rip_ctx, "ripple_peak_index", area_a, area_b, dlt_ctx
    )

if unreadable_lfp:
    print(f"{len(unreadable_lfp)} LFP files could not be read and were skipped:")
    for msg in unreadable_lfp:
        print("   ", msg)

fig, axes = plt.subplots(
    1,
    len(cross_area_diffs),
    figsize=(4.5 * len(cross_area_diffs), 4.5),
    subplot_kw={"projection": "polar"},
)
axes = np.atleast_1d(axes)
for ax, (pair, diffs) in zip(axes, cross_area_diffs.items()):
    circular_hist(ax, diffs, color="tab:teal")
    if len(diffs):
        R, z, p = rayleigh_test(diffs)
        ax.set_title(f"{pair[0]}-{pair[1]}\nRayleigh R={R:.3f}, p={p:.4f}", fontsize=9)
    else:
        ax.set_title(f"{pair[0]}-{pair[1]}\n(no data yet)", fontsize=9)
plt.tight_layout()
save_fig(fig, "17_cross_area_delta_phase_diff")
plt.show()

# Can each day be trusted? (first hour)

## Circular shift per recording day (first hour)

Ripple train shifted within each recording's NREM time, all seven rats. Use the z-score, p can't go below 1/N_ITER.

In [ ]:
CACHE_PERDAY = OUT / "perday_shift_cache_first_hour"
CACHE_PERDAY.mkdir(parents=True, exist_ok=True)


def per_day_shift_zscores(dlt_df, rip_df, area, rats, n_iter=N_ITER, seed=0):
    """Per-day circular-shift test: for each rat-day, shift the ripple train
    within its own recording (scope="recording") and compare the observed
    R->D / D->R window counts against that day's own shuffled null.
    Cached per area so an interrupted run can resume without redoing days
    already computed."""
    cache_f = CACHE_PERDAY / f"zscores_{area}_n{n_iter}.csv"
    if cache_f.exists():
        return pd.read_csv(cache_f, parse_dates=["date"])

    d_area = dlt_df[dlt_df["area"] == area]
    left_ = edges[:-1]
    win_rd = (left_ >= -250) & (left_ < -50)
    win_dr = (left_ >= 50) & (left_ < 400)
    tail = (left_ < -800) | (left_ >= 800)

    rows = []
    for rat in rats:
        d_rat = d_area[d_area["rat"] == rat]
        r_rat = rip_df[rip_df["rat"] == rat]
        for date, dg in d_rat.groupby("date"):
            rg = r_rat[r_rat["date"] == date]
            if rg.empty or dg.empty:
                continue
            dist = sg.surrogate_distribution(
                dg,
                rg,
                "delta",
                "ripple",
                edges,
                n_iter=n_iter,
                seed=seed,
                scope="recording",
                verbose=False,
            )
            rows.append(
                {
                    "rat": rat,
                    "date": date,
                    "area": area,
                    "z_window_RD": float(np.nanmean(dist["z"][win_rd])),
                    "z_window_DR": float(np.nanmean(dist["z"][win_dr])),
                    "z_tail": float(np.nanmean(dist["z"][tail])),
                    "n_bouts": dist["n_bouts"],
                }
            )
    out = pd.DataFrame(rows)
    out.to_csv(cache_f, index=False)
    return out


perday_z = pd.concat(
    [per_day_shift_zscores(dlt_ctx, rip_ctx, area, SHIFT_RATS) for area in AREAS],
    ignore_index=True,
)
print(f"{len(perday_z)} rat-day-area z-scores computed (first hour)")
print("z_tail (should sit near zero -- the null-calibration check):")
print(perday_z.groupby("area")["z_tail"].agg(["mean", "std"]).round(3).to_string())

## Per-day significance plots (R->D, first hour)

In [ ]:
fig, axes = plt.subplots(1, len(AREAS), figsize=(4.8 * len(AREAS), 4), sharey=True)
axes = np.atleast_1d(axes)
for ax, area in zip(axes, AREAS):
    z = perday_z[perday_z["area"] == area]["z_window_RD"].dropna()
    ax.hist(z, bins=30, color="tab:blue", alpha=0.75)
    ax.axvline(2, color="k", lw=1, ls=":", label="z = 2")
    ax.set_title(f"{area}   {int((z >= 2).sum())}/{len(z)} days z>=2")
    ax.set_xlabel("z_window_RD")
axes[0].set_ylabel("rat-days")
axes[0].legend(fontsize=8)
fig.suptitle("Per-day R->D significance, all seven rats -- first hour", fontsize=12)
plt.tight_layout()
save_fig(fig, "13_perday_zwindow_hist")
plt.show()

## D->R per-day significance plots (first hour)

Same for D->R (`z_window_DR`).

In [ ]:
fig, axes = plt.subplots(1, len(AREAS), figsize=(4.8 * len(AREAS), 4), sharey=True)
axes = np.atleast_1d(axes)
for ax, area in zip(axes, AREAS):
    z = perday_z[perday_z["area"] == area]["z_window_DR"].dropna()
    ax.hist(z, bins=30, color="tab:orange", alpha=0.75)
    ax.axvline(2, color="k", lw=1, ls=":", label="z = 2")
    ax.set_title(f"{area}   {int((z >= 2).sum())}/{len(z)} days z>=2")
    ax.set_xlabel("z_window_DR")
axes[0].set_ylabel("rat-days")
axes[0].legend(fontsize=8)
fig.suptitle("Per-day D->R significance, all seven rats -- first hour", fontsize=12)
plt.tight_layout()
save_fig(fig, "13b_perday_zwindow_hist_DR")
plt.show()

# NREM duration cutoff for the trend statistics (first hour)

At least 10 min of NREM within hour 1 (`day_nrem_hour1`), only for the per-rat trends.

This removes more rat-days than in the full dataset, see the counts below.

Also applied to every over-time plot below (trajectories, block plots, absolute counts, ripple-anchored proportions); the removed rat-days are listed in red under each plot.

In [ ]:
MIN_NREM_MIN = 10  # minutes of NREM sleep required for a rat-day to count (first hour)

good_days = day_nrem_hour1[day_nrem_hour1 >= MIN_NREM_MIN].reset_index()[
    ["rat", "date"]
]

print(
    f"rat-days: {day_nrem_hour1.shape[0]} total -> {len(good_days)} with >= {MIN_NREM_MIN} min "
    f"NREM in the first hour ({day_nrem_hour1.shape[0] - len(good_days)} dropped)"
)

daily_nrem = daily.merge(good_days, on=["rat", "date"])

print("rat-days remaining per area, before vs after the NREM-duration cutoff:")
before = daily.groupby("area").size()
after = daily_nrem.groupby("area").size()
print(
    pd.DataFrame(
        {"before": before, "after": after, "dropped": before - after}
    ).to_string()
)

# rat-days removed by the cutoff, listed under every plot from here on
day_group = rip_ctx[["rat", "date", "group"]].drop_duplicates(["rat", "date"])
nrem_removed = (
    day_nrem_hour1[day_nrem_hour1 < MIN_NREM_MIN]
    .rename("nrem_min")
    .reset_index()
    .merge(day_group, on=["rat", "date"], how="left")
    .sort_values(["rat", "date"])
    .reset_index(drop=True)
)
print(f"removed by the NREM cutoff: {len(nrem_removed)} rat-days")


def add_nrem_footnote(fig, rats=None, groups=("HC", "Training")):
    """List the rat-days removed by the NREM cutoff under the figure."""
    rem = nrem_removed[nrem_removed["group"].isin(groups)]
    if rats is not None:
        rem = rem[rem["rat"].isin(rats)]
    if rem.empty:
        return
    parts = []
    for rat, g in rem.groupby("rat"):
        days = ", ".join(
            f"{pd.Timestamp(d).date()} {grp} ({m:.1f} min)"
            for d, grp, m in zip(g["date"], g["group"], g["nrem_min"])
        )
        parts.append(f"Rat {rat}: {days}")
    note = f"Removed, < {MIN_NREM_MIN} min NREM in hour 1: " + " | ".join(parts)
    fig.text(
        0.5,
        -0.01,
        note,
        ha="center",
        va="top",
        fontsize=8,
        color="firebrick",
        wrap=True,
    )

# Trends over time per rat (first hour)

## Every recording day per rat (NREM duration filtered, first hour)

One panel per rat, x = days since first recording, separate HC and training trend lines.

In [ ]:
for measure in MEASURES:
    for area in AREAS:
        sub = daily_nrem[daily_nrem["area"] == area]
        rats = sorted(sub["rat"].unique())
        ncol = 3
        nrow = int(np.ceil(len(rats) / ncol))
        fig, axes = plt.subplots(
            nrow, ncol, figsize=(4.8 * ncol, 3.9 * nrow), sharey=True, squeeze=False
        )

        for ax, rat in zip(axes.ravel(), rats):
            r = sub[sub["rat"] == rat].sort_values("days_elapsed")
            smax = max(r["n_seq"].max(), 1)
            bits = []

            for grp, colour in [("HC", "tab:orange"), ("Training", "tab:blue")]:
                g = r[r["group"] == grp]
                if g.empty:
                    continue
                ax.scatter(
                    g["days_elapsed"],
                    g[measure],
                    s=g["n_seq"] / smax * 80 + 15,
                    alpha=0.8,
                    color=colour,
                    edgecolor="white",
                    linewidth=0.5,
                    label=grp,
                    zorder=3,
                )
                if len(g) >= 4:
                    b, a = np.polyfit(g["days_elapsed"], g[measure], 1)
                    xs = np.array([g["days_elapsed"].min(), g["days_elapsed"].max()])
                    ax.plot(xs, a + b * xs, color=colour, lw=2.2, alpha=0.9, zorder=2)
                    rho, pv = spearmanr(g["days_elapsed"], g[measure])
                    bits.append(f"{grp}: rho={rho:+.2f}{'*' if pv < 0.05 else ''}")

            if measure.startswith("ratio"):
                ax.axhline(1, color="k", lw=1, ls=":", zorder=1)
            ax.set_title(
                f"rat {rat} ({area})   n={len(r)}\n" + "   ".join(bits), fontsize=9
            )
            ax.set_xlabel("days since first recording")
            ax.grid(alpha=0.25, lw=0.6)

        for ax in axes.ravel()[len(rats) :]:
            ax.axis("off")
        for row in axes:
            row[0].set_ylabel(measure)
        axes[0][0].legend(fontsize=8, loc="lower right")
        fig.suptitle(
            f"{area} — {measure} over time, first hour (NREM-filtered)"
            "   (dot size = sequences; * = p < 0.05 for that line)",
            fontsize=13,
        )
        plt.tight_layout()
        add_nrem_footnote(fig, rats=rats)
        save_fig(fig, f"01_{measure}_{area}")
        plt.show()

### The numbers behind the points

In [ ]:
cols = [
    "rat",
    "date",
    "days_elapsed",
    "group",
    "n_seq",
    "n_ripple",
    "nrem_min",
    "rate_RD",
    "ratio_RD",
    "prop_ripples",
    "rate_DR",
    "ratio_DR",
]
tab = (
    daily_nrem[daily_nrem["area"] == AREAS[0]]
    .assign(date=lambda d: pd.to_datetime(d["date"]).dt.strftime("%d/%m/%y"))
    .sort_values(["rat", "days_elapsed"])[cols]
)
print(f"=== {AREAS[0]} (first hour, NREM-filtered) ===")
print(tab.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

print("totals per rat, all areas (first hour, NREM-filtered):")
print(
    daily_nrem.groupby(["area", "rat"])
    .agg(
        days=("date", "nunique"),
        total_seq=("n_seq", "sum"),
        median_seq=("n_seq", "median"),
    )
    .to_string()
)

# Trend statistics (first hour)
## Per rat, Spearman plus a permutation on day order

Descriptive only.

In [ ]:
from modules.event_coupling import overtime_stats as ot

AXES = {
    "days_elapsed": "days since first recording",
    "gl_num": "goal location",
    "session_num": "session within block",
}

train_daily = daily_nrem[daily_nrem["group"] == "Training"]

per_rat = []
for axis in AXES:
    src = daily_nrem if axis == "days_elapsed" else train_daily
    for measure in MEASURES:
        t = ot.trend_per_rat(
            src,
            axis,
            y_col=measure,
            group_cols=("area",),
            min_points=5,
            n_iter=1000,
            seed=0,
        )
        if len(t):
            per_rat.append(t.assign(axis=axis, measure=measure))
per_rat = pd.concat(per_rat, ignore_index=True)

print(
    per_rat[["axis", "measure", "area", "rat", "n_days", "rho", "p_perm"]]
    .round(3)
    .to_string(index=False)
)

## Across rats, Wilcoxon then FDR (first hour)

FDR within each direction (R->D 27 tests, D->R 18). `n_days`/`n_rats` show how many rat-days are left after the NREM cutoff.

In [ ]:
trend_summary = []
for (axis, measure), g in per_rat.groupby(["axis", "measure"]):
    res = ot.across_rats(g, group_cols=("area",), verbose=False)
    trend_summary.append(res.assign(axis=axis, measure=measure))
trend_summary = pd.concat(trend_summary, ignore_index=True)

DIRECTION = {
    "rate_RD": "R->D",
    "ratio_RD": "R->D",
    "prop_ripples": "R->D",
    "rate_DR": "D->R",
    "ratio_DR": "D->R",
}
trend_summary["direction"] = trend_summary["measure"].map(DIRECTION)

trend_summary["p_fdr"] = np.nan
for d, g in trend_summary.groupby("direction"):
    trend_summary.loc[g.index, "p_fdr"] = ot.fdr(g["p_wilcoxon"])

trend_summary["p_fdr_all"] = ot.fdr(trend_summary["p_wilcoxon"])

trend_summary = trend_summary.sort_values(["direction", "p_wilcoxon"])

cols = [
    "direction",
    "axis",
    "measure",
    "area",
    "n_rats",
    "median_rho",
    "n_positive",
    "p_wilcoxon",
    "p_fdr",
    "p_fdr_all",
]
if "p_note" in trend_summary.columns:
    cols.append("p_note")
print(trend_summary[cols].round(4).to_string(index=False))

if "p_note" in trend_summary.columns:
    flagged = trend_summary[trend_summary["p_note"] != ""]
    print(f"tests with an unreliable p-value: {len(flagged)} of {len(trend_summary)}")
    if len(flagged):
        print(
            flagged[["direction", "axis", "measure", "area", "p_wilcoxon", "p_note"]]
            .round(4)
            .to_string(index=False)
        )

for d, g in trend_summary.groupby("direction"):
    n_sig = int((g["p_fdr"] < 0.05).sum())
    n_sig2 = int((g["p_fdr_all"] < 0.05).sum())
    print(
        f"{d}: {len(g)} tests | {n_sig} survive FDR within direction "
        f"| {n_sig2} survive FDR across all {len(trend_summary)}"
    )

sig = trend_summary[trend_summary["p_fdr"] < 0.05]
if len(sig):
    print(
        sig[
            [
                "direction",
                "axis",
                "measure",
                "area",
                "median_rho",
                "n_positive",
                "p_fdr",
                "p_fdr_all",
            ]
        ]
        .round(4)
        .to_string(index=False)
    )
else:
    print("nothing survives FDR in either scope.")

### What the numbers say, in words (first hour)

In [ ]:
for (axis, measure, area), g in per_rat.groupby(["axis", "measure", "area"]):
    n = len(g)
    sig = int((g["p_perm"] < 0.05).sum())
    up = int((g["rho"] > 0).sum())
    if sig == 0:
        continue
    row = trend_summary[
        (trend_summary["axis"] == axis)
        & (trend_summary["measure"] == measure)
        & (trend_summary["area"] == area)
    ]
    p_fdr = float(row["p_fdr"].iloc[0]) if len(row) else np.nan
    agree = "SAME direction" if up in (0, n) else f"{up} up / {n - up} down"
    print(
        f"{area:>4} {measure:<13} {axis:<13}: {sig}/{n} rats significant "
        f"| {agree:<16} | across-rat p_fdr {p_fdr:.3f}"
    )

print("A rat can trend strongly on its own and still contribute nothing to the")
print("group result, if the other rats trend the opposite way.")

# Trajectories and within-block view (first hour)

## Trajectories, not only correlations (first hour)

Unfiltered `daily` table. Panels: raw, divided by first value, Spearman rho.

In [ ]:
COHORTS = {
    "build-up (rats 1-8)": [r for r in RATS if r <= 8],
    "update (rats 9-16)": [r for r in RATS if r > 8],
}


def trajectory_figure(measure, area):
    sub = daily_nrem[daily_nrem["area"] == area]
    if sub.empty:
        return

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)

    for ax, (label, rats) in zip(axes, COHORTS.items()):
        for rat in rats:
            r = sub[sub["rat"] == rat].sort_values("days_elapsed")
            if len(r) < 3:
                continue
            base = r[measure].iloc[:3].mean()
            if base == 0 or np.isnan(base):
                continue
            y = r[measure] / base
            (line,) = ax.plot(
                r["days_elapsed"], y, lw=0.9, alpha=0.35, marker="o", ms=2.5
            )
            if len(r) >= 5:
                b, a = np.polyfit(r["days_elapsed"], y, 1)
                xs = np.array([r["days_elapsed"].min(), r["days_elapsed"].max()])
                rho, pv = spearmanr(r["days_elapsed"], y)
                ax.plot(
                    xs,
                    a + b * xs,
                    lw=2.6,
                    color=line.get_color(),
                    label=f"rat {rat}   \u03c1={rho:+.2f}{'*' if pv < 0.05 else ''}",
                )
            else:
                line.set_label(f"rat {rat}")

        ax.axhline(1, color="k", lw=1.2, ls=":")
        ax.set_xlabel("days since first recording")
        ax.set_title(label, fontsize=11)
        ax.legend(fontsize=9, loc="best")
        ax.grid(alpha=0.25, lw=0.6)

    axes[0].set_ylabel(f"{measure} / that rat's first 3 days")
    fig.suptitle(
        f"{area} \u2014 {measure} over time, first hour   "
        "(faint = daily values, bold = fitted trend; * = p < 0.05)",
        fontsize=13,
    )
    plt.tight_layout()
    add_nrem_footnote(fig)
    save_fig(fig, f"02_trajectories_{measure}_{area}")
    plt.show()


def axis_figure(measure, area):
    sub = daily_nrem[(daily_nrem["area"] == area) & (daily_nrem["group"] == "Training")]
    if sub.empty:
        return

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
    for ax, (axis, xlabel) in zip(
        axes, [("gl_num", "goal location"), ("session_num", "session within block")]
    ):
        for rat in RATS:
            r = sub[sub["rat"] == rat].dropna(subset=[axis])
            if len(r) < 3:
                continue
            g = r.groupby(axis)[measure].mean()
            base = g.iloc[:2].mean()
            if base == 0 or np.isnan(base):
                continue
            ax.plot(
                g.index,
                g.values / base,
                lw=1.8,
                alpha=0.85,
                marker="o",
                ms=4,
                label=f"rat {rat}",
            )
        ax.axhline(1, color="k", lw=1.2, ls=":")
        ax.set_xlabel(xlabel)
        ax.grid(alpha=0.25, lw=0.6)
    axes[0].set_ylabel(f"{measure} / that rat's first values")
    axes[0].legend(fontsize=8, ncol=2)
    fig.suptitle(
        f"{area} \u2014 {measure} by goal location and session, first hour   "
        "(training days only; no median \u2014 cohorts differ in length)",
        fontsize=12,
    )
    plt.tight_layout()
    add_nrem_footnote(fig, groups=["Training"])
    save_fig(fig, f"03_axes_{measure}_{area}")
    plt.show()


for measure in MEASURES:
    for area in AREAS:
        trajectory_figure(measure, area)

for area in AREAS:
    axis_figure("ratio_RD", area)

In [ ]:
def block_figure(measure, area):
    sub = daily_nrem[(daily_nrem["area"] == area) & (daily_nrem["group"] == "Training")]
    sub = sub.dropna(subset=["gl_num", "session_num"])
    if sub.empty:
        return

    sub = sub.copy()
    sub["step"] = sub["gl_num"] * 10 + sub["session_num"]
    order = sorted(sub["step"].unique())
    pos = {s: i for i, s in enumerate(order)}
    labels = [f"GL{int(s // 10)}_S{int(s % 10)}" for s in order]

    fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), sharey=True)
    for ax, (label, rats) in zip(axes, COHORTS.items()):
        for rat in rats:
            r = sub[sub["rat"] == rat]
            if len(r) < 3:
                continue
            g = r.groupby("step")[measure].mean()
            base = g.iloc[:2].mean()
            if base == 0 or np.isnan(base):
                continue
            ax.plot(
                [pos[s] for s in g.index],
                g.values / base,
                lw=1.6,
                alpha=0.85,
                marker="o",
                ms=4,
                label=f"rat {rat}",
            )
        for i, s in enumerate(order):
            if int(s // 10) % 2 == 0:
                ax.axvspan(i - 0.5, i + 0.5, color="0.92", zorder=0)
        ax.axhline(1, color="k", lw=1.2, ls=":")
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=90, fontsize=7)
        ax.set_title(label, fontsize=11)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.2, lw=0.5, axis="y")
    axes[0].set_ylabel(f"{measure} / that rat's first values")
    fig.suptitle(
        f"{area} — {measure} across every training day in order, first hour   "
        "(shading marks goal-location blocks)",
        fontsize=12,
    )
    plt.tight_layout()
    add_nrem_footnote(fig, groups=["Training"])
    save_fig(fig, f"04_blocks_{measure}_{area}")
    plt.show()


for area in AREAS:
    block_figure("ratio_RD", area)

In [ ]:
AREA_COLOUR = {"HPC": "#D64545", "PL": "#E0A800", "RSC": "#3E8E5A"}


def rat_block_figure(measure, rat):
    d = daily_nrem[(daily_nrem["rat"] == rat) & (daily_nrem["group"] == "Training")]
    d = d.dropna(subset=[measure, "gl_num", "session_num"]).copy()
    if d.empty:
        return

    d["step"] = d["gl_num"] * 100 + d["session_num"]
    order = sorted(d["step"].unique())
    pos = {s: i for i, s in enumerate(order)}
    labels = [f"GL{int(s // 100)}_S{int(s % 100)}" for s in order]
    d["x"] = d["step"].map(pos)

    fig, ax = plt.subplots(figsize=(max(9, 0.55 * len(order)), 5))

    tilt = {}
    for area in AREAS:
        a = d[d["area"] == area].sort_values("x")
        if a.empty:
            continue
        g = a.groupby("x")[measure].mean()
        ax.plot(
            g.index,
            g.values,
            lw=1.1,
            alpha=0.35,
            marker="o",
            ms=4.5,
            color=AREA_COLOUR[area],
            zorder=2,
        )

        ups = 0
        total = 0
        for gl in sorted(a["gl_num"].unique()):
            b = a[a["gl_num"] == gl].groupby("x")[measure].mean()
            if len(b) < 3:
                continue
            m, c = np.polyfit(b.index, b.values, 1)
            xs = np.array([b.index.min(), b.index.max()])
            ax.plot(
                xs,
                c + m * xs,
                lw=3.0,
                color=AREA_COLOUR[area],
                alpha=0.95,
                solid_capstyle="round",
                zorder=4,
            )
            total += 1
            ups += m > 0
        tilt[area] = (ups, total)

    for s in order:
        if int(s // 100) % 2 == 0:
            ax.axvspan(pos[s] - 0.5, pos[s] + 0.5, color="0.94", zorder=0)
    if measure.startswith("ratio"):
        ax.axhline(1, color="k", lw=1, ls=":", zorder=1)

    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90, fontsize=9)
    ax.set_xlim(-0.6, len(labels) - 0.4)
    ax.set_ylabel(measure)
    ax.grid(alpha=0.22, lw=0.6, axis="y")

    handles = [
        plt.Line2D(
            [],
            [],
            color=AREA_COLOUR[a],
            lw=3,
            label=f"{a}   {tilt[a][0]}/{tilt[a][1]} blocks up" if a in tilt else a,
        )
        for a in AREAS
    ]
    ax.legend(handles=handles, fontsize=10, loc="best", framealpha=0.9)

    ax.set_title(
        f"rat {rat} — {measure} within each goal location, first hour\n"
        "faint = daily values,  bold = trend within that block",
        fontsize=12,
    )
    plt.tight_layout()
    add_nrem_footnote(fig, rats=[rat], groups=["Training"])
    save_fig(fig, f"06_blocks_{measure}_rat{rat}")
    plt.show()


for rat in sorted(daily_nrem["rat"].unique()):
    rat_block_figure("ratio_RD", rat)

# Absolute-count block plots (first hour)

Raw counts, per-cohort x-axis.

In [ ]:
def get_shared_ylim(measure, area):
    gl_vals = daily_nrem[
        (daily_nrem["area"] == area) & (daily_nrem["group"] == "Training")
    ][measure]
    hc_vals = daily_nrem[(daily_nrem["area"] == area) & (daily_nrem["group"] == "HC")][
        measure
    ]
    all_vals = pd.concat([gl_vals, hc_vals]).dropna()
    if all_vals.empty:
        return None
    pad = 0.05 * (all_vals.max() - all_vals.min())
    return (all_vals.min() - pad, all_vals.max() + pad)


def block_figure_count(measure, area, ylim=None):
    sub_all = daily_nrem[
        (daily_nrem["area"] == area) & (daily_nrem["group"] == "Training")
    ]
    sub_all = sub_all.dropna(subset=["gl_num", "session_num"])
    if sub_all.empty:
        return

    fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), sharey=True)
    for ax, (label, rats) in zip(axes, COHORTS.items()):
        sub = sub_all[sub_all["rat"].isin(rats)].copy()
        if sub.empty:
            ax.set_title(label, fontsize=11)
            continue

        sub["step"] = sub["gl_num"] * 10 + sub["session_num"]
        order = sorted(sub["step"].unique())
        pos = {s: i for i, s in enumerate(order)}
        labels = [f"GL{int(s // 10)}_S{int(s % 10)}" for s in order]

        for rat in rats:
            r = sub[sub["rat"] == rat]
            if len(r) < 3:
                continue
            g = r.groupby("step")[measure].mean().reindex(order)
            xs = [pos[s] for s in order]
            ax.plot(
                xs, g.values, lw=1.6, alpha=0.85, marker="o", ms=4, label=f"rat {rat}"
            )

        for i, s in enumerate(order):
            if int(s // 10) % 2 == 0:
                ax.axvspan(i - 0.5, i + 0.5, color="0.92", zorder=0)
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=90, fontsize=7)
        ax.set_title(label, fontsize=11)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.2, lw=0.5, axis="y")

    axes[0].set_ylabel("absolute count")
    if ylim:
        axes[0].set_ylim(ylim)
    fig.suptitle(
        f"{area} \u2014 {measure} across every training day in order, first hour   "
        "(shading marks goal-location blocks)",
        fontsize=12,
    )
    plt.tight_layout()
    add_nrem_footnote(fig, groups=["Training"])
    save_fig(fig, f"04_blocks_{measure}_{area}")
    plt.show()


def hc_block_figure(measure, area, ylim=None):
    sub_all = daily_nrem[
        (daily_nrem["area"] == area) & (daily_nrem["group"] == "HC")
    ].copy()
    if sub_all.empty:
        return

    fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), sharey=True)
    for ax, (label, rats) in zip(axes, COHORTS.items()):
        sub = sub_all[sub_all["rat"].isin(rats)].copy()
        if sub.empty:
            ax.set_title(label, fontsize=11)
            continue

        max_hc = 0
        for rat in rats:
            r = sub[sub["rat"] == rat].sort_values("date")
            if len(r) < 2:
                continue
            hc_index = np.arange(1, len(r) + 1)
            max_hc = max(max_hc, len(r))
            ax.plot(
                hc_index,
                r[measure].values,
                lw=1.6,
                alpha=0.85,
                marker="o",
                ms=4,
                label=f"rat {rat}",
            )

        for i in range(1, max_hc + 1):
            if i % 2 == 0:
                ax.axvspan(i - 0.5, i + 0.5, color="0.92", zorder=0)
        ax.set_xticks(range(1, max_hc + 1))
        ax.set_xticklabels(
            [f"HC{i}" for i in range(1, max_hc + 1)], rotation=90, fontsize=7
        )
        ax.set_title(label, fontsize=11)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.2, lw=0.5, axis="y")

    axes[0].set_ylabel("absolute count")
    if ylim:
        axes[0].set_ylim(ylim)
    fig.suptitle(
        f"{area} \u2014 {measure} across home-cage days, in order, first hour   "
        "(shading is cosmetic, alternating every other HC session)",
        fontsize=12,
    )
    plt.tight_layout()
    add_nrem_footnote(fig, groups=["HC"])
    save_fig(fig, f"05_hc_{measure}_{area}")
    plt.show()

## R->D, absolute count (first hour)

In [ ]:
daily_nrem["n_seq_RD"] = daily_nrem["n_seq"]
for area in AREAS:
    ylim = get_shared_ylim("n_seq_RD", area)
    block_figure_count("n_seq_RD", area, ylim=ylim)
    hc_block_figure("n_seq_RD", area, ylim=ylim)

## D->R, absolute count (first hour)

In [ ]:
for area in AREAS:
    ylim = get_shared_ylim("n_seq_DR", area)
    block_figure_count("n_seq_DR", area, ylim=ylim)
    hc_block_figure("n_seq_DR", area, ylim=ylim)

# Ripple-anchored proportion, normalized to the home-cage baseline (first hour)

Ripple-anchored proportions on the first-hour tables.

`_hcnorm` falls back to the second-nearest HC day when the nearest is 0 (happens more often in the first hour). Rat-days still dropped are printed.

In [ ]:
def count_ripples_with_partner(rip, dlt, lag_lo, lag_hi):
    """Unique ripples with a delta wave in the given lag window, per rat-day.

    lag = ripple_peak - delta_peak (project convention: negative = ripple first)
      Ripple->Delta  (ripple leads):  window [-250, -50]
      Delta->Ripple, ripple-anchored (delta leads, but counted per ripple):
                     window [+50, +400]
    """
    rows = []
    for (rat, date), rg in rip.groupby(["rat", "date"]):
        dg = dlt[(dlt["rat"] == rat) & (dlt["date"] == date)]
        n_coupled = 0
        for bout in rg["nrem_bout_index"].unique():
            r_b = rg.loc[rg["nrem_bout_index"] == bout, "ripple_peak_index"].values
            d_b = (
                dg.loc[dg["nrem_bout_index"] == bout, "delta_peak_index"].values
                if not dg.empty
                else np.array([])
            )
            if len(r_b) == 0 or len(d_b) == 0:
                continue
            lags = r_b[:, None] - d_b[None, :]
            has_partner = ((lags >= lag_lo) & (lags < lag_hi)).any(axis=1)
            n_coupled += int(has_partner.sum())
        rows.append(
            {"rat": rat, "date": date, "n_coupled": n_coupled, "n_ripple": len(rg)}
        )
    return pd.DataFrame(rows)


def _synth_partner(n_days=3, seed=0):
    rng = np.random.default_rng(seed)
    D, R = [], []
    for day in range(1, n_days + 1):
        d = np.sort(rng.uniform(0, 20000, 60))
        r = np.sort(rng.uniform(0, 20000, 10))
        if day == 2:
            take = rng.choice(len(d), 4, replace=False)
            r = np.sort(np.concatenate([r, d[take] + 140]))
        if day == 3:
            take = rng.choice(len(d), 4, replace=False)
            r = np.sort(np.concatenate([r, d[take] - 140]))
        r = r[(r > 0) & (r < 20000)]
        D.append(
            pd.DataFrame(
                {
                    "rat": 1,
                    "date": f"day{day}",
                    "nrem_bout_index": 0,
                    "delta_peak_index": d,
                }
            )
        )
        R.append(
            pd.DataFrame(
                {
                    "rat": 1,
                    "date": f"day{day}",
                    "nrem_bout_index": 0,
                    "ripple_peak_index": r,
                }
            )
        )
    return pd.concat(D, ignore_index=True), pd.concat(R, ignore_index=True)


_dlt, _rip = _synth_partner()

_rd = count_ripples_with_partner(_rip, _dlt, -250, -50)
assert _rd.loc[_rd.date == "day2", "n_coupled"].iloc[0] >= 4
assert (
    _rd.loc[_rd.date == "day2", "n_coupled"].iloc[0]
    > _rd.loc[_rd.date == "day1", "n_coupled"].iloc[0]
)
print("Ripple->Delta self-test PASSED")

_dr = count_ripples_with_partner(_rip, _dlt, 50, 400)
assert _dr.loc[_dr.date == "day3", "n_coupled"].iloc[0] >= 4
assert (
    _dr.loc[_dr.date == "day3", "n_coupled"].iloc[0]
    > _dr.loc[_dr.date == "day1", "n_coupled"].iloc[0]
)
print("Delta->Ripple (ripple-anchored) self-test PASSED")

In [ ]:
def nearest_hc_normalize(df, value_col):
    """For each training day, divide by the nearest home-cage day's value
    (same rat, same area). If the nearest HC day is 0, falls back to the
    second-nearest (capped at two attempts). If that is also 0, the
    training day is dropped and reported in `dropped`."""
    out_rows, dropped = [], []
    for (rat, area), g in df.groupby(["rat", "area"]):
        hc = g[g["group"] == "HC"].sort_values("days_elapsed")
        train = g[g["group"] == "Training"]
        if hc.empty or train.empty:
            continue
        for _, row in train.iterrows():
            hc_sorted = hc.iloc[
                (hc["days_elapsed"] - row["days_elapsed"]).abs().argsort()
            ]
            used = None
            for rank, (_, hc_row) in enumerate(hc_sorted.iterrows()):
                if rank >= 2:
                    break
                if hc_row[value_col] and hc_row[value_col] > 0:
                    used = hc_row
                    break
            if used is None:
                nearest_two = hc_sorted.head(2)
                dropped.append(
                    {
                        "rat": rat,
                        "area": area,
                        "date": row["date"],
                        "days_elapsed": row["days_elapsed"],
                        "nearest_hc_days_elapsed": list(nearest_two["days_elapsed"]),
                    }
                )
                continue
            out_rows.append(
                {
                    **row.to_dict(),
                    "ratio": row[value_col] / used[value_col],
                    "nearest_hc_days_elapsed": used["days_elapsed"],
                }
            )
    return pd.DataFrame(out_rows), dropped


_test_df = pd.DataFrame(
    [
        {"rat": 1, "area": "HPC", "group": "HC", "days_elapsed": 5, "prop": 0.0},
        {"rat": 1, "area": "HPC", "group": "HC", "days_elapsed": 10, "prop": 0.30},
        {"rat": 1, "area": "HPC", "group": "HC", "days_elapsed": 20, "prop": 0.40},
        {"rat": 1, "area": "HPC", "group": "Training", "days_elapsed": 6, "prop": 0.60},
    ]
)
_res, _drop = nearest_hc_normalize(_test_df, "prop")
assert _res["nearest_hc_days_elapsed"].iloc[0] == 10, (
    "fallback to second-nearest did not fire"
)
assert abs(_res["ratio"].iloc[0] - 0.60 / 0.30) < 1e-9
assert len(_drop) == 0
print("Nearest-HC-session (with fallback) self-test PASSED")


def avg_hc_normalize(df, value_col):
    """For each training day, divide by THAT RAT's own average value across
    ALL its HC days (same rat, same area)."""
    out_rows = []
    for (rat, area), g in df.groupby(["rat", "area"]):
        hc = g[g["group"] == "HC"]
        train = g[g["group"] == "Training"]
        if hc.empty or train.empty:
            continue
        hc_mean = hc[value_col].mean()
        if not hc_mean or hc_mean <= 0 or np.isnan(hc_mean):
            continue
        for _, row in train.iterrows():
            out_rows.append(
                {
                    **row.to_dict(),
                    "ratio": row[value_col] / hc_mean,
                    "hc_mean": hc_mean,
                    "n_hc_days": len(hc),
                }
            )
    return pd.DataFrame(out_rows)


_test_df2 = pd.DataFrame(
    [
        {"rat": 1, "area": "HPC", "group": "HC", "days_elapsed": 5, "prop": 0.20},
        {"rat": 1, "area": "HPC", "group": "HC", "days_elapsed": 10, "prop": 0.30},
        {"rat": 1, "area": "HPC", "group": "HC", "days_elapsed": 20, "prop": 0.40},
        {
            "rat": 1,
            "area": "HPC",
            "group": "Training",
            "days_elapsed": 12,
            "prop": 0.60,
        },
    ]
)
_res2 = avg_hc_normalize(_test_df2, "prop")
_expected_mean = np.mean([0.20, 0.30, 0.40])
assert abs(_res2["hc_mean"].iloc[0] - _expected_mean) < 1e-9
assert abs(_res2["ratio"].iloc[0] - 0.60 / _expected_mean) < 1e-9
print("Average-HC-normalization self-test PASSED")

## Ripple->Delta: ripple-anchored proportion (first hour)

Fraction of ripples with a delta 50-250 ms later.

In [ ]:
prop_rd_rows = []
for area in AREAS:
    d = dlt_ctx[dlt_ctx["area"] == area]
    for (rat, date), rg in rip_ctx.groupby(["rat", "date"]):
        dg = d[(d["rat"] == rat) & (d["date"] == date)]
        t = count_ripples_with_partner(
            rg.assign(date=date), dg.assign(date=date), -250, -50
        )
        if t.empty or t["n_ripple"].iloc[0] == 0:
            continue
        row = rip_ctx[(rip_ctx["rat"] == rat) & (rip_ctx["date"] == date)].iloc[0]
        prop_rd_rows.append(
            {
                "rat": rat,
                "date": date,
                "area": area,
                "prop_ripple_RD": t["n_coupled"].iloc[0] / t["n_ripple"].iloc[0],
                "group": row["group"],
                "days_elapsed": row["days_elapsed"],
                "gl_num": row.get("gl_num"),
                "session_num": row.get("session_num"),
            }
        )

prop_rd = pd.DataFrame(prop_rd_rows)
print(f"{len(prop_rd)} rat-day-area rows (first hour)")
print(prop_rd.groupby(["area", "group"])["prop_ripple_RD"].median().to_string())

## Delta->Ripple: ripple-anchored proportion (first hour)

Fraction of ripples with a delta 50-400 ms before.

In [ ]:
prop_dr_rows = []
for area in AREAS:
    d = dlt_ctx[dlt_ctx["area"] == area]
    for (rat, date), rg in rip_ctx.groupby(["rat", "date"]):
        dg = d[(d["rat"] == rat) & (d["date"] == date)]
        t = count_ripples_with_partner(
            rg.assign(date=date), dg.assign(date=date), 50, 400
        )
        if t.empty or t["n_ripple"].iloc[0] == 0:
            continue
        row = rip_ctx[(rip_ctx["rat"] == rat) & (rip_ctx["date"] == date)].iloc[0]
        prop_dr_rows.append(
            {
                "rat": rat,
                "date": date,
                "area": area,
                "prop_ripple_DR": t["n_coupled"].iloc[0] / t["n_ripple"].iloc[0],
                "group": row["group"],
                "days_elapsed": row["days_elapsed"],
                "gl_num": row.get("gl_num"),
                "session_num": row.get("session_num"),
            }
        )

prop_dr = pd.DataFrame(prop_dr_rows)
print(f"{len(prop_dr)} rat-day-area rows (first hour)")
print(prop_dr.groupby(["area", "group"])["prop_ripple_DR"].median().to_string())

## Ripple-in-Spindle: ripple-anchored proportion (first hour)

Fraction of ripples inside a spindle.

In [ ]:
spindles = add_condition(
    build_wave(ROOT, "spindle", SPINDLE_METHOD, PHASES, WAVE_CACHE, rebuild=False),
    SESSION_MAP_CSV,
)

spn = spindles[
    (spindles["rat"].isin(RATS))
    & (spindles["group"] != "other")
    & (spindles["area"].isin(AREAS))
]
spn = add_context(spn)
spn_full = spn
spn = filter_first_hour_fixed(spn_full, "spindle_peak_index")

print(
    f"{len(spn_full):,} spindles total -> {len(spn):,} in true first hour "
    f"({100 * len(spn) / len(spn_full):.1f}%)"
)
print(spn.groupby("area").size().to_string())

In [ ]:
spn_ctx = spn.merge(
    rip_ctx[["rat", "date", "days_elapsed"]].drop_duplicates(),
    on=["rat", "date"],
    how="left",
)

print(
    spn_ctx.groupby(["rat", "group"])["date"]
    .nunique()
    .unstack(fill_value=0)
    .to_string()
)

In [ ]:
from modules.event_coupling import spindle_ripple as sr

RULE = "any_overlap"  # slide 7 / Navarro Lobato: partial overlaps count

prop_sp_rows = []
for area in AREAS:
    s = spn_ctx[spn_ctx["area"] == area]
    for (rat, date), rg in rip_ctx.groupby(["rat", "date"]):
        sg_ = s[(s["rat"] == rat) & (s["date"] == date)]
        if sg_.empty or len(rg) == 0:
            continue
        t = sr.count_overlap(sg_, rg, rule=RULE)
        n_rip = t["n_ripple"].sum()
        if n_rip == 0:
            continue
        row = rip_ctx[(rip_ctx["rat"] == rat) & (rip_ctx["date"] == date)].iloc[0]
        prop_sp_rows.append(
            {
                "rat": rat,
                "date": date,
                "area": area,
                "prop_ripple_spindle": t["n_coupled"].sum() / n_rip,
                "group": row["group"],
                "days_elapsed": row["days_elapsed"],
                "gl_num": row.get("gl_num"),
                "session_num": row.get("session_num"),
            }
        )

prop_sp = pd.DataFrame(prop_sp_rows)
print(f"{len(prop_sp)} rat-day-area rows (first hour)")
print(prop_sp.groupby(["area", "group"])["prop_ripple_spindle"].median().to_string())

## Normalize each measure to a home-cage baseline, both ways (first hour)

`_hcnorm` = nearest HC session (second-nearest if 0), `_avghcnorm` = rat's average over HC days.

In [ ]:
# NREM cutoff, as for the trend statistics
prop_rd = prop_rd.merge(good_days, on=["rat", "date"])
prop_dr = prop_dr.merge(good_days, on=["rat", "date"])
prop_sp = prop_sp.merge(good_days, on=["rat", "date"])

prop_rd_hcnorm, dropped_rd = nearest_hc_normalize(prop_rd, "prop_ripple_RD")
prop_dr_hcnorm, dropped_dr = nearest_hc_normalize(prop_dr, "prop_ripple_DR")
prop_sp_hcnorm, dropped_sp = nearest_hc_normalize(prop_sp, "prop_ripple_spindle")

prop_rd_avghcnorm = avg_hc_normalize(prop_rd, "prop_ripple_RD")
prop_dr_avghcnorm = avg_hc_normalize(prop_dr, "prop_ripple_DR")
prop_sp_avghcnorm = avg_hc_normalize(prop_sp, "prop_ripple_spindle")

for name, hcn, avgn, dropped in [
    ("Ripple->Delta", prop_rd_hcnorm, prop_rd_avghcnorm, dropped_rd),
    ("Delta->Ripple", prop_dr_hcnorm, prop_dr_avghcnorm, dropped_dr),
    ("Ripple-in-Spindle", prop_sp_hcnorm, prop_sp_avghcnorm, dropped_sp),
]:
    print(
        f"{name}: {len(hcn)} days normalized to nearest first-hour HC session "
        f"({len(dropped)} dropped -- both nearest tries were 0) | "
        f"{len(avgn)} days normalized to that rat's own average HC"
    )

all_dropped = dropped_rd + dropped_dr + dropped_sp
if all_dropped:
    print("=== rat-days dropped from _hcnorm (both nearest-HC tries were 0) ===")
    for d in all_dropped:
        print(
            f"  rat {d['rat']} {d['area']}, {d['date']} (days_elapsed={d['days_elapsed']}): "
            f"nearest two HC days {d['nearest_hc_days_elapsed']} both had 0"
        )

## Plots (first hour)

In [ ]:
def block_figure_normalized(df, area, label, suffix, title_phrase):
    sub_all = df[df["area"] == area].dropna(subset=["gl_num", "session_num"])
    if sub_all.empty:
        return

    fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), sharey=True)
    for ax, (cohort_label, rats) in zip(axes, COHORTS.items()):
        sub = sub_all[sub_all["rat"].isin(rats)].copy()
        if sub.empty:
            ax.set_title(cohort_label, fontsize=11)
            continue
        sub["step"] = sub["gl_num"] * 10 + sub["session_num"]
        order = sorted(sub["step"].unique())
        pos = {s: i for i, s in enumerate(order)}
        tick_labels = [f"GL{int(s // 10)}_S{int(s % 10)}" for s in order]

        for rat in rats:
            r = sub[sub["rat"] == rat]
            if len(r) < 3:
                continue
            g = r.groupby("step")["ratio"].mean().reindex(order)
            ax.plot(
                [pos[s] for s in order],
                g.values,
                lw=1.6,
                alpha=0.85,
                marker="o",
                ms=4,
                label=f"rat {rat}",
            )

        for i, s in enumerate(order):
            if int(s // 10) % 2 == 0:
                ax.axvspan(i - 0.5, i + 0.5, color="0.92", zorder=0)
        ax.axhline(1, color="k", lw=1.2, ls=":")
        ax.set_xticks(range(len(tick_labels)))
        ax.set_xticklabels(tick_labels, rotation=90, fontsize=7)
        ax.set_title(cohort_label, fontsize=11)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.2, lw=0.5, axis="y")

    axes[0].set_ylabel("ratio")
    fig.suptitle(
        f"{area} \u2014 normalized to {title_phrase}, first hour   "
        "(shading marks goal-location blocks)",
        fontsize=12,
    )
    plt.tight_layout()
    add_nrem_footnote(fig)
    save_fig(fig, f"ratio_{suffix}_{label}_{area}")
    plt.show()


def get_shared_ylim_prop(df, measure, area):
    train_vals = df[(df["area"] == area) & (df["group"] == "Training")][measure]
    hc_vals = df[(df["area"] == area) & (df["group"] == "HC")][measure]
    all_vals = pd.concat([train_vals, hc_vals]).dropna()
    if all_vals.empty:
        return None
    pad = 0.05 * (all_vals.max() - all_vals.min())
    return (all_vals.min() - pad, all_vals.max() + pad)


for measure_col, daily_df, hcn_df, avgn_df, label in [
    ("prop_ripple_RD", prop_rd, prop_rd_hcnorm, prop_rd_avghcnorm, "RippleToDelta"),
    ("prop_ripple_DR", prop_dr, prop_dr_hcnorm, prop_dr_avghcnorm, "DeltaToRipple"),
    (
        "prop_ripple_spindle",
        prop_sp,
        prop_sp_hcnorm,
        prop_sp_avghcnorm,
        "RippleInSpindle",
    ),
]:
    print(f"=== {label} ===")
    for area in AREAS:
        ylim = get_shared_ylim_prop(daily_df, measure_col, area)
        block_figure_normalized(
            hcn_df,
            area,
            label,
            "normalized_hcnorm",
            "nearest first-hour home-cage session",
        )
        block_figure_normalized(
            avgn_df,
            area,
            label,
            "normalized_avghcnorm",
            "that rat's own average first-hour HC",
        )

---
# Notes

## Open points

- The hour-1 NREM cutoff removes more rat-days than in the full dataset.